In [ ]:
%pip install -q "transformers>=4.41.0" datasets accelerate scikit-learn pandas numpy underthesea sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 55.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 38.5 MB/s eta 0:00:00


In [ ]:
import inspect
import json
import math
import os
import random
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from datasets import Dataset, DatasetDict
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.utils.class_weight import compute_class_weight
from torch import nn
from tqdm.auto import tqdm
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)
from underthesea import word_tokenize

os.environ["WANDB_DISABLED"] = "true"
os.environ["TOKENIZERS_PARALLELISM"] = "false"


def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


SEED = 42
set_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
GPU: Tesla T4


## 1. Mount Google Drive va cau hinh duong dan

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive")

DATA_RELATIVE_PATH = Path("data/processed/student_voice_enriched_reviewed.csv")
PHOBERT_CACHE_RELATIVE_PATH = Path("data/processed/student_voice_enriched_reviewed_phobert.csv")

PROJECT_DIR_CANDIDATES = [
    DRIVE_ROOT / "Student Voice Intelligence",
    DRIVE_ROOT / "AI thuc chien" / "Student Voice Intelligence",
    DRIVE_ROOT / "Colab Notebooks" / "Student Voice Intelligence",
]

PROJECT_DIR = None
for candidate in PROJECT_DIR_CANDIDATES:
    if (candidate / DATA_RELATIVE_PATH).exists():
        PROJECT_DIR = candidate
        break

DATA_PATH = PROJECT_DIR / DATA_RELATIVE_PATH
PHOBERT_CACHE_PATH = PROJECT_DIR / PHOBERT_CACHE_RELATIVE_PATH

MODEL_ROOT = PROJECT_DIR / "outputs/models/transformer"
REPORT_DIR = PROJECT_DIR / "outputs/reports/transformer/topic_phobertv2"
MODEL_ROOT.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

RUN_NAME = f"phobert_base_v2_topic_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
RUN_DIR = MODEL_ROOT / RUN_NAME
CHECKPOINT_DIR = RUN_DIR / "checkpoints"
MODEL_DIR = RUN_DIR / "model"

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_DIR:", PROJECT_DIR)
print("DATA_PATH:", DATA_PATH)
print("PHOBERT_CACHE_PATH:", PHOBERT_CACHE_PATH)
print("MODEL_DIR:", MODEL_DIR)
print("REPORT_DIR:", REPORT_DIR)

Mounted at /content/drive
PROJECT_DIR: /content/drive/MyDrive/Student Voice Intelligence
DATA_PATH: /content/drive/MyDrive/Student Voice Intelligence/data/processed/student_voice_enriched_reviewed.csv
PHOBERT_CACHE_PATH: /content/drive/MyDrive/Student Voice Intelligence/data/processed/student_voice_enriched_reviewed_phobert.csv
MODEL_DIR: /content/drive/MyDrive/Student Voice Intelligence/outputs/models/transformer/phobert_base_v2_topic_20260621_092559/model
REPORT_DIR: /content/drive/MyDrive/Student Voice Intelligence/outputs/reports/transformer/topic_phobertv2


## 2. Cau hinh train


In [ ]:
MODEL_NAME = "vinai/phobert-base-v2"
TEXT_COL = "text"
TEXT_PHOBERT_COL = "text_phobert"
TARGET_COL = "topic_group"
SPLIT_COL = "split_original"

TOPIC_LABELS = [
    "career_jobs",
    "events_news",
    "facilities",
    "others",
    "personal_social",
    "spam",
    "student_services",
    "teaching_learning",
]

label2id = {label: idx for idx, label in enumerate(TOPIC_LABELS)}
id2label = {idx: label for label, idx in label2id.items()}

MAX_LENGTH = 192
TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 32
NUM_EPOCHS = 3
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
USE_CLASS_WEIGHTS = True

BASELINE_TOPIC_TEST_ACCURACY = 0.815421
BASELINE_TOPIC_TEST_MACRO_F1 = 0.658308

print("label2id:", label2id)

label2id: {'career_jobs': 0, 'events_news': 1, 'facilities': 2, 'others': 3, 'personal_social': 4, 'spam': 5, 'student_services': 6, 'teaching_learning': 7}


## 3. Load data va tao/doc cache text da tach tu


In [ ]:
df = pd.read_csv(DATA_PATH, low_memory=False)

required_cols = [TEXT_COL, TARGET_COL, SPLIT_COL]
missing_cols = [col for col in required_cols if col not in df.columns]

df[TEXT_COL] = df[TEXT_COL].fillna("").astype(str).str.strip()


def segment_for_phobert(text: str) -> str:
    text = " ".join(str(text).split())
    if not text:
        return ""
    return word_tokenize(text, format="text")


use_cache = False
if PHOBERT_CACHE_PATH.exists():
    cached_df = pd.read_csv(PHOBERT_CACHE_PATH, low_memory=False)
    cache_is_valid = (
        len(cached_df) == len(df)
        and TEXT_PHOBERT_COL in cached_df.columns
        and TEXT_COL in cached_df.columns
    )
    if cache_is_valid:
        df[TEXT_PHOBERT_COL] = cached_df[TEXT_PHOBERT_COL].fillna("").astype(str)
        use_cache = True

if not use_cache:
    tqdm.pandas(desc="Word segmentation for PhoBERT")
    df[TEXT_PHOBERT_COL] = df[TEXT_COL].progress_apply(segment_for_phobert)
    PHOBERT_CACHE_PATH.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(PHOBERT_CACHE_PATH, index=False, encoding="utf-8-sig")

display(df[[TEXT_COL, TEXT_PHOBERT_COL, TARGET_COL, SPLIT_COL]].head())

,text,text_phobert,topic_group,split_original
0,em xin chào các anh chị em em là sinh viên vừa...,em xin chào các anh_chị_em em là sinh_viên vừa...,teaching_learning,train
1,xây dựng mô hình sư phạm chuẩn mực sáng tạo ti...,xây_dựng mô_hình sư_phạm chuẩn_mực sáng_tạo ti...,teaching_learning,train
2,sao lại ghét cái kiểu làm sai xong khóc lóc ăn...,sao lại ghét cái kiểu làm sai xong khóc_lóc ăn...,others,train
3,chào thầy đăng ký học ghép . môn học hiện hệ t...,chào thầy đăng_ký học ghép . môn_học hiện hệ_t...,teaching_learning,train
4,con bé vẫn hoang mang lắm .,con_bé vẫn hoang_mang lắm .,others,train


## 4. Kiem tra label va chuan bi dataframe train

In [ ]:
work_df = df[[TEXT_COL, TEXT_PHOBERT_COL, TARGET_COL, SPLIT_COL]].copy()
work_df[TEXT_COL] = work_df[TEXT_COL].fillna("").astype(str).str.strip()
work_df[TEXT_PHOBERT_COL] = work_df[TEXT_PHOBERT_COL].fillna("").astype(str).str.strip()
work_df[TARGET_COL] = work_df[TARGET_COL].fillna("").astype(str).str.strip()
work_df[SPLIT_COL] = work_df[SPLIT_COL].fillna("").astype(str).str.strip()

unknown_labels = sorted(set(work_df[TARGET_COL].dropna()) - set(TOPIC_LABELS) - {""})

work_df = work_df[
    (work_df[TEXT_COL] != "")
    & (work_df[TEXT_PHOBERT_COL] != "")
    & (work_df[TARGET_COL].isin(TOPIC_LABELS))
    & (work_df[SPLIT_COL].isin(["train", "validation", "test"]))
].copy()

work_df["labels"] = work_df[TARGET_COL].map(label2id).astype(int)

print("Shape:", work_df.shape)
print("Topic distribution by split:")
display(pd.crosstab(work_df[SPLIT_COL], work_df[TARGET_COL], margins=True))

print("Overall topic distribution:")
display(work_df[TARGET_COL].value_counts().rename_axis("topic_group").reset_index(name="count"))

Shape: (49141, 5)
Topic distribution by split:


topic_group,career_jobs,events_news,facilities,others,personal_social,spam,student_services,teaching_learning,All
split_original,,,,,,,,,
test,163,316,145,3041,453,84,611,4966,9779
train,564,1090,497,10640,1568,281,2112,17722,34474
validation,81,158,70,1537,226,40,305,2471,4888
All,808,1564,712,15218,2247,405,3028,25159,49141


Overall topic distribution:


,topic_group,count
0,teaching_learning,25159
1,others,15218
2,student_services,3028
3,personal_social,2247
4,events_news,1564
5,career_jobs,808
6,facilities,712
7,spam,405


## 5. Tinh class weights cho topic


In [7]:
train_labels = work_df.loc[work_df[SPLIT_COL] == "train", "labels"].to_numpy()
class_weight_values = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(len(TOPIC_LABELS)),
    y=train_labels,
)
class_weights = torch.tensor(class_weight_values, dtype=torch.float32)

class_weight_df = pd.DataFrame(
    {
        "label_id": np.arange(len(TOPIC_LABELS)),
        "topic_group": TOPIC_LABELS,
        "class_weight": class_weight_values,
        "train_count": [int((train_labels == idx).sum()) for idx in range(len(TOPIC_LABELS))],
    }
)

display(class_weight_df)
class_weight_df.to_csv(REPORT_DIR / "class_weights.csv", index=False, encoding="utf-8-sig")

,label_id,topic_group,class_weight,train_count
0,0,career_jobs,7.640514,564
1,1,events_news,3.953440,1090
2,2,facilities,8.670523,497
3,3,others,0.405005,10640
4,4,personal_social,2.748246,1568
5,5,spam,15.335409,281
6,6,student_services,2.040365,2112
7,7,teaching_learning,0.243158,17722


## 6. Tao Hugging Face Dataset va tokenize

In [8]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


def to_hf_dataset(split_name: str) -> Dataset:
    split_df = work_df.loc[work_df[SPLIT_COL] == split_name, [TEXT_PHOBERT_COL, "labels"]].reset_index(drop=True)
    if split_df.empty:
        raise ValueError(f"Split rong: {split_name}")
    return Dataset.from_pandas(split_df, preserve_index=False)


raw_datasets = DatasetDict(
    {
        "train": to_hf_dataset("train"),
        "validation": to_hf_dataset("validation"),
        "test": to_hf_dataset("test"),
    }
)


def tokenize_batch(batch):
    return tokenizer(
        batch[TEXT_PHOBERT_COL],
        truncation=True,
        max_length=MAX_LENGTH,
    )


tokenized_datasets = raw_datasets.map(tokenize_batch, batched=True)
tokenized_datasets = tokenized_datasets.remove_columns([TEXT_PHOBERT_COL])

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print(tokenized_datasets)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/678 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/895k [00:00<?, ?B/s]

bpe.codes:   0%|          | 0.00/1.14M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.13M [00:00<?, ?B/s]

Map:   0%|          | 0/34474 [00:00<?, ? examples/s]

Map:   0%|          | 0/4888 [00:00<?, ? examples/s]

Map:   0%|          | 0/9779 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 34474
    })
    validation: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 4888
    })
    test: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 9779
    })
})


## 7. Khoi tao model, metric va weighted trainer

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(TOPIC_LABELS),
    id2label=id2label,
    label2id=label2id,
)


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro"),
        "weighted_f1": f1_score(labels, preds, average="weighted"),
    }


class WeightedLossTrainer(Trainer):

    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")

        if self.class_weights is not None:
            weights = self.class_weights.to(device=logits.device, dtype=logits.dtype)
            loss_fct = nn.CrossEntropyLoss(weight=weights)
        else:
            loss_fct = nn.CrossEntropyLoss()

        loss = loss_fct(logits.view(-1, model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

pytorch_model.bin:   0%|          | 0.00/540M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: vinai/phobert-base-v2
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.dense.bias      | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## 8. Train

In [10]:
steps_per_epoch = math.ceil(len(tokenized_datasets["train"]) / TRAIN_BATCH_SIZE)
warmup_steps = int(steps_per_epoch * NUM_EPOCHS * 0.06)

training_args_kwargs = {
    "output_dir": str(CHECKPOINT_DIR),
    "learning_rate": LEARNING_RATE,
    "per_device_train_batch_size": TRAIN_BATCH_SIZE,
    "per_device_eval_batch_size": EVAL_BATCH_SIZE,
    "num_train_epochs": NUM_EPOCHS,
    "weight_decay": WEIGHT_DECAY,
    "warmup_steps": warmup_steps,
    "max_grad_norm": 1.0,
    "logging_steps": 50,
    "save_strategy": "epoch",
    "save_total_limit": 2,
    "load_best_model_at_end": True,
    "metric_for_best_model": "macro_f1",
    "greater_is_better": True,
    "fp16": torch.cuda.is_available(),
    "report_to": "none",
    "seed": SEED,
}

training_args_signature = inspect.signature(TrainingArguments.__init__)
if "eval_strategy" in training_args_signature.parameters:
    training_args_kwargs["eval_strategy"] = "epoch"
else:
    training_args_kwargs["evaluation_strategy"] = "epoch"

training_args = TrainingArguments(**training_args_kwargs)

trainer_kwargs = {
    "model": model,
    "args": training_args,
    "train_dataset": tokenized_datasets["train"],
    "eval_dataset": tokenized_datasets["validation"],
    "data_collator": data_collator,
    "compute_metrics": compute_metrics,
    "class_weights": class_weights if USE_CLASS_WEIGHTS else None,
}

trainer_signature = inspect.signature(Trainer.__init__)
if "processing_class" in trainer_signature.parameters:
    trainer_kwargs["processing_class"] = tokenizer
elif "tokenizer" in trainer_signature.parameters:
    trainer_kwargs["tokenizer"] = tokenizer

trainer = WeightedLossTrainer(**trainer_kwargs)

train_result = trainer.train()
trainer.save_model(str(MODEL_DIR))
tokenizer.save_pretrained(str(MODEL_DIR))

model.safetensors:   0%|          | 0.00/540M [00:00<?, ?B/s]

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Weighted F1
1,0.697461,0.723947,0.814648,0.688256,0.819749
2,0.697717,0.726582,0.825900,0.718983,0.829669
3,0.462949,0.760139,0.830810,0.720270,0.834301


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/drive/MyDrive/Student Voice Intelligence/outputs/models/transformer/phobert_base_v2_topic_20260621_092559/model/tokenizer_config.json',
 '/content/drive/MyDrive/Student Voice Intelligence/outputs/models/transformer/phobert_base_v2_topic_20260621_092559/model/vocab.txt',
 '/content/drive/MyDrive/Student Voice Intelligence/outputs/models/transformer/phobert_base_v2_topic_20260621_092559/model/bpe.codes',
 '/content/drive/MyDrive/Student Voice Intelligence/outputs/models/transformer/phobert_base_v2_topic_20260621_092559/model/added_tokens.json')

## 9. Evaluate validation/test va luu report

In [11]:
def evaluate_split(split_name: str) -> dict:
    predictions = trainer.predict(tokenized_datasets[split_name])
    y_true = predictions.label_ids
    y_pred = np.argmax(predictions.predictions, axis=-1)

    metrics = {
        "split": split_name,
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro"),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted"),
    }

    report_df = pd.DataFrame(
        classification_report(
            y_true,
            y_pred,
            target_names=TOPIC_LABELS,
            output_dict=True,
            zero_division=0,
        )
    ).T
    cm_df = pd.DataFrame(confusion_matrix(y_true, y_pred), index=TOPIC_LABELS, columns=TOPIC_LABELS)

    report_df.to_csv(REPORT_DIR / f"{split_name}_classification_report.csv", encoding="utf-8-sig")
    cm_df.to_csv(REPORT_DIR / f"{split_name}_confusion_matrix.csv", encoding="utf-8-sig")

    return metrics


validation_metrics = evaluate_split("validation")
test_metrics = evaluate_split("test")

all_metrics = {
    "model_name": MODEL_NAME,
    "target_col": TARGET_COL,
    "text_col_used": TEXT_PHOBERT_COL,
    "labels": TOPIC_LABELS,
    "max_length": MAX_LENGTH,
    "train_batch_size": TRAIN_BATCH_SIZE,
    "eval_batch_size": EVAL_BATCH_SIZE,
    "num_epochs": NUM_EPOCHS,
    "learning_rate": LEARNING_RATE,
    "warmup_steps": warmup_steps,
    "use_class_weights": USE_CLASS_WEIGHTS,
    "class_weights": {label: float(weight) for label, weight in zip(TOPIC_LABELS, class_weight_values)},
    "baseline_topic_test_accuracy": BASELINE_TOPIC_TEST_ACCURACY,
    "baseline_topic_test_macro_f1": BASELINE_TOPIC_TEST_MACRO_F1,
    "validation": validation_metrics,
    "test": test_metrics,
    "model_dir": str(MODEL_DIR),
    "report_dir": str(REPORT_DIR),
    "phobert_cache_path": str(PHOBERT_CACHE_PATH),
}

with open(REPORT_DIR / "metrics.json", "w", encoding="utf-8") as f:
    json.dump(all_metrics, f, ensure_ascii=False, indent=2)

with open(MODEL_ROOT / "phobert_base_v2_topic_latest.txt", "w", encoding="utf-8") as f:
    f.write(str(MODEL_DIR))

print(json.dumps(all_metrics, ensure_ascii=False, indent=2))
print("Reports:", REPORT_DIR)

{
  "model_name": "vinai/phobert-base-v2",
  "target_col": "topic_group",
  "text_col_used": "text_phobert",
  "labels": [
    "career_jobs",
    "events_news",
    "facilities",
    "others",
    "personal_social",
    "spam",
    "student_services",
    "teaching_learning"
  ],
  "max_length": 192,
  "train_batch_size": 16,
  "eval_batch_size": 32,
  "num_epochs": 3,
  "learning_rate": 2e-05,
  "warmup_steps": 387,
  "use_class_weights": true,
  "class_weights": {
    "career_jobs": 7.640514184397163,
    "events_news": 3.953440366972477,
    "facilities": 8.670523138832998,
    "others": 0.4050046992481203,
    "personal_social": 2.7482461734693877,
    "spam": 15.33540925266904,
    "student_services": 2.0403645833333335,
    "teaching_learning": 0.24315822141970433
  },
  "baseline_topic_test_accuracy": 0.815421,
  "baseline_topic_test_macro_f1": 0.658308,
  "validation": {
    "split": "validation",
    "accuracy": 0.830810147299509,
    "macro_f1": 0.7202697509519858,
    "weigh

## 10. Test nhanh vai cau thuc te

In [12]:
infer_tokenizer = AutoTokenizer.from_pretrained(str(MODEL_DIR))
infer_model = AutoModelForSequenceClassification.from_pretrained(str(MODEL_DIR)).to(DEVICE)
infer_model.eval()


def prepare_phobert_texts(texts):
    if isinstance(texts, str):
        texts = [texts]
    return [segment_for_phobert(text) for text in texts]


def predict_topic(texts):
    if isinstance(texts, str):
        texts = [texts]

    phobert_texts = prepare_phobert_texts(texts)
    inputs = infer_tokenizer(
        phobert_texts,
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt",
    ).to(DEVICE)

    with torch.no_grad():
        logits = infer_model(**inputs).logits
        probs = torch.softmax(logits, dim=-1).cpu().numpy()

    rows = []
    for text, phobert_text, prob in zip(texts, phobert_texts, probs):
        pred_id = int(np.argmax(prob))
        row = {
            "text": text,
            "text_phobert": phobert_text,
            "prediction": id2label[pred_id],
            "confidence": float(prob[pred_id]),
        }
        for idx, label in id2label.items():
            row[f"prob_{label}"] = float(prob[idx])
        rows.append(row)

    return pd.DataFrame(rows)


sample_texts = [
    "Wifi phòng học quá yếu, máy chiếu bị mờ nên rất khó học.",
    "Giảng viên giảng khó hiểu, bài tập quá nhiều và chấm điểm không rõ ràng.",
    "Phòng đào tạo phản hồi email quá chậm, em không biết hỏi ai.",
    "Em muốn hỏi thông tin tuyển dụng thực tập và ngày hội việc làm.",
    "Câu lạc bộ tổ chức sự kiện rất bổ ích cho sinh viên.",
    "Bài đăng này bị spam và không liên quan đến sinh viên.",
    "Bạn đó có hành vi xúc phạm sinh viên trên mạng xã hội.",
    "Thông báo mới về lịch nghỉ lễ và kế hoạch học tập.",
]

predict_topic(sample_texts)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

,text,text_phobert,prediction,confidence,prob_career_jobs,prob_events_news,prob_facilities,prob_others,prob_personal_social,prob_spam,prob_student_services,prob_teaching_learning
0,"Wifi phòng học quá yếu, máy chiếu bị mờ nên rấ...","Wifi phòng học quá yếu , máy_chiếu bị mờ nên r...",facilities,0.996355,0.000581,0.000437,0.996355,0.000320,0.000385,0.000547,0.000427,0.000949
1,"Giảng viên giảng khó hiểu, bài tập quá nhiều v...","Giảng_viên giảng khó hiểu , bài_tập quá nhiều ...",teaching_learning,0.986586,0.001491,0.001521,0.004507,0.003064,0.001180,0.000618,0.001033,0.986586
2,"Phòng đào tạo phản hồi email quá chậm, em khôn...","Phòng đào_tạo phản_hồi email quá chậm , em khô...",student_services,0.578330,0.002327,0.016082,0.182684,0.028029,0.001748,0.009592,0.578330,0.181208
3,Em muốn hỏi thông tin tuyển dụng thực tập và n...,Em muốn hỏi thông_tin tuyển_dụng thực_tập và n...,career_jobs,0.987084,0.987084,0.002888,0.001924,0.001182,0.001401,0.002822,0.001276,0.001423
4,Câu lạc bộ tổ chức sự kiện rất bổ ích cho sinh...,Câu_lạc_bộ tổ_chức sự_kiện rất bổ_ích cho sinh...,events_news,0.944993,0.001632,0.944993,0.003936,0.004885,0.003799,0.002122,0.004155,0.034478
5,Bài đăng này bị spam và không liên quan đến si...,Bài đăng này bị spam và không liên_quan đến si...,others,0.517059,0.002803,0.002579,0.005220,0.517059,0.011561,0.010961,0.008834,0.440984
6,Bạn đó có hành vi xúc phạm sinh viên trên mạng...,Bạn đó có hành_vi xúc_phạm sinh_viên trên mạng...,teaching_learning,0.635116,0.002088,0.006973,0.004655,0.199872,0.132579,0.003833,0.014884,0.635116
7,Thông báo mới về lịch nghỉ lễ và kế hoạch học ...,Thông_báo mới về lịch nghỉ lễ và kế_hoạch học_...,events_news,0.510468,0.003234,0.510468,0.012576,0.059127,0.006669,0.003643,0.011339,0.392943
